# evaluations_v2: test-set evaluation through the shared pipeline

The rules: the test set is only scored, nothing is fitted on it, and there are no threshold, contamination or hyperparameter changes. Decisions come from each model's own `predict()`, i.e. the thresholds set at training time by `contamination=0.1` / `nu=0.1`.

**Paste the printed output of every cell.** Section 6 also writes `results_v2.json` next to this notebook.

In [ ]:
import sys, json, pickle, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd, sklearn

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    while True:
        if (cur / "Python").is_dir() and (cur / "RustSniffer").is_dir():
            return cur
        if cur.parent == cur:
            raise RuntimeError("Could not find NIDS root (folder containing Python/ and RustSniffer/).")
        cur = cur.parent

NB_DIR       = Path.cwd()                                   # Python/Notebooks/src
PROJECT_ROOT = find_project_root(NB_DIR)
SRC_DIR      = PROJECT_ROOT / "Python" / "src"
MODEL_DIR    = SRC_DIR / "models"                           # the ONLY artefact dir v2 uses (detector.py loads from here)
DATA_DIR     = NB_DIR / "datasets"
TRAIN_CSV    = DATA_DIR / "UNSW_NB15_training_set.csv"
TEST_CSV     = DATA_DIR / "UNSW_NB15_testing_set.csv"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from preprocessing import FeaturePipeline, FEATURE_COLS, CAT_COLS, SchemaError

def sha12(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()[:12] if p.exists() else "MISSING"

print("python", platform.python_version(), "| sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR   :", MODEL_DIR)
for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl", "feature_meta.json"]:
    print(f"  {f:26s} {sha12(MODEL_DIR / f)}")
OLD_DIR = PROJECT_ROOT / "Python" / "Notebooks" / "Python" / "models"   # where v1 cell 8 / evaluations.ipynb also read and wrote
if OLD_DIR.exists():
    print("Second model dir exists (v1 leftover):", OLD_DIR)
    for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl"]:
        print(f"  {f:26s} {sha12(OLD_DIR / f)}")

from sklearn.metrics import confusion_matrix, classification_report, f1_score
pipe = FeaturePipeline.load(MODEL_DIR)       # fails loudly on a v1 (integer-code) contract
models = {}
for n, f in [("iforest", "isolation_forest.pkl"), ("lof", "local_outlier_factor.pkl"), ("ocsvm", "one_class_svm.pkl")]:
    with open(MODEL_DIR / f, "rb") as fh: models[n] = pickle.load(fh)

df_test = pd.read_csv(TEST_CSV)
y = df_test["label"].astype(int).to_numpy()
cat = df_test["attack_cat"].astype(str).str.strip()
print("test", df_test.shape, "| label counts", dict(zip(*np.unique(y, return_counts=True))))
print("attack_cat counts:"); print(cat.value_counts().to_string())

## 1. Shared-pipeline input

In [ ]:
X, unknown = pipe.prepare_frame(df_test)
print("X", X.shape, "| test labels not in training vocabulary:", unknown)
for c in CAT_COLS:
    unseen = sorted(set(df_test[c].astype(str)) - set(pipe.vocab[c]))
    print(f"  {c}: unseen labels {unseen}")

## 2. Per-model results

In [ ]:
def report(name, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    print("=" * 64); print(name)
    print(f"TN={tn} FP={fp} FN={fn} TP={tp} | FPR(normal)={fp/(tn+fp):.4f} | TPR(attack)={tp/(tp+fn):.4f}")
    print(classification_report(y, pred, digits=4, zero_division=0))
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                weighted_f1=float(f1_score(y, pred, average="weighted", zero_division=0)),
                macro_f1=float(f1_score(y, pred, average="macro", zero_division=0)),
                accuracy=float((tp + tn) / len(y)))

preds, results = {}, {}
for n, m in models.items():
    preds[n] = (m.predict(X) == -1).astype(int)
    results[n] = report(n, preds[n])

## 3. Ensemble: 2-of-3 majority vote (same rule as `detector.detect`)

In [ ]:
votes = sum(preds.values())
preds["ensemble_2of3"] = (votes >= 2).astype(int)
results["ensemble_2of3"] = report("ensemble_2of3 (majority vote)", preds["ensemble_2of3"])
print("rows by number of attack votes (rows = true label):")
print(pd.crosstab(pd.Series(y, name="label"), pd.Series(votes, name="attack_votes")).to_string())

## 4. Detection rate per attack category (and flag rate on Normal)

In [ ]:
per_cat = pd.DataFrame({n: pd.Series(p).groupby(cat.values).mean() for n, p in preds.items()})
per_cat.insert(0, "n", cat.value_counts())
print(per_cat.sort_values("n", ascending=False).round(4).to_string())

## 5. R1 / R2 / R3 reconciliation: replay the exact v1 `evaluations.ipynb` paths
- **v1 cell 17**: `LabelEncoder` fit on the test set, then `model.predict(X)` with **X unscaled** (Isolation Forest only), giving R1/R2
- **v1 cell 23**: same encoding, then `scaler.transform`, then all three models, giving the R3 bar chart
- **v2**: shared pipeline (section 2)

In [ ]:
from sklearn.preprocessing import LabelEncoder
X_v1 = df_test[FEATURE_COLS].copy()
for col in X_v1.columns:
    if not pd.api.types.is_numeric_dtype(X_v1[col]):
        X_v1[col] = LabelEncoder().fit_transform(X_v1[col].astype(str))
X_v1 = X_v1.to_numpy(dtype=float)

with open(MODEL_DIR / "scaler.pkl", "rb") as f: scaler = pickle.load(f)
X_v1_scaled = scaler.transform(X_v1)
X_v2_raw, _ = pipe.encode_frame(df_test)
for j, c in enumerate(FEATURE_COLS):
    if c in CAT_COLS:
        print(f"{c}: rows where v1 test-fit LabelEncoder code != training code: {(X_v1[:, j] != X_v2_raw[:, j]).mean():.4f}")

rows = []
p = (models["iforest"].predict(X_v1) == -1).astype(int)
tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
rows.append(["v1 cell17: unscaled, test-fit LE", "iforest", f1_score(y, p, average="weighted", zero_division=0), tn, fp, fn, tp])
for n, m in models.items():
    p = (m.predict(X_v1_scaled) == -1).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    rows.append(["v1 cell23: scaled, test-fit LE", n, f1_score(y, p, average="weighted", zero_division=0), tn, fp, fn, tp])
for n in models:
    r = results[n]; rows.append(["v2 shared pipeline", n, r["weighted_f1"], r["tn"], r["fp"], r["fn"], r["tp"]])
recon = pd.DataFrame(rows, columns=["path", "model", "weighted_f1", "TN", "FP", "FN", "TP"])
print(recon.round(4).to_string(index=False))

## 6. Save a machine-readable record

In [ ]:
from datetime import datetime, timezone
record = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
    "artefact_sha12": {f: sha12(MODEL_DIR / f) for f in ["scaler.pkl", "isolation_forest.pkl", "local_outlier_factor.pkl", "one_class_svm.pkl", "feature_meta.json"]},
    "test_csv_sha12": sha12(TEST_CSV),
    "unknown_labels": unknown,
    "results": results,
    "per_category": per_cat.round(6).to_dict(),
    "reconciliation": recon.to_dict(orient="records"),
}
out = NB_DIR / "results_v2.json"
out.write_text(json.dumps(record, indent=2, default=float))
print("wrote", out)